In [1]:
import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai

In [2]:
# Load the documents
documents = []
docDir = "RAGdocs"
for filename in os.listdir(docDir):
    if filename.endswith(".pdf"):
        with open(os.path.join(docDir, filename), "rb") as f:
            filepath = os.path.join(docDir, filename)

            reader = PdfReader(f)
            text = ""
            for page in reader.pages:
                text += page.extract_text() + "\n"
            documents.append((filepath, text))

document_texts = [text for filepath, text in documents]

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)


Code to show how many documents I embedded:

In [3]:
print(len(documents))

12


In [4]:
# Define model and embed the documents
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

In [5]:
def rag_query(query, documents, embeddings):
    query_embedding = model.encode([query])

    similarities = model.similarity(query_embedding, embeddings)
    most_similar_index = similarities[0].argmax()
    most_similar_doc = documents[most_similar_index]

    model_query = f"""
        The user asked the following question: {query}

        Here is the document that most closely matches the question:
        <document>
        {most_similar_doc[1]}
        </document>

        Based on <document>, provide an answer to the user's question.
        """

    client = genai.Client(api_key="mykey")
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config=genai.types.GenerateContentConfig(
            system_instruction="You are Cameron's assistant and help answering questions about his documents."
        ),
        contents=model_query
    )
    print(response.text)

This function takes three arguments: a user query, a set of documents, and the embeddings of the documents. We first create an embedding of the user query and compare it to the embeddings of the documents to find the document that best matches the user query. Then, a model query is generated, telling the model to use the user query and answer it using the document identified by the similarity function. The function prints the response to the query generated by Gemini.

In [6]:
rag_query("Why did I want to work for the Oklahoma City Thunder?", documents, embeddings)

Cameron wanted to work for the Oklahoma City Thunder because he initially became interested in data science due to its use in sports, and basketball has always been his favorite sport. As a lifelong NBA fan, he was excited by the opportunity to apply his professional skills to help a well-run franchise succeed. Additionally, he saw it as an excellent opportunity to learn from expert data scientists, gain valuable experience, and work on projects in an industry he loves.


In [7]:
rag_query("Where has Cameron worked, and what did he do at each job?", documents, embeddings)

Cameron has worked at the following places and held these positions:

*   **Financial Planning and Analytics Intern** at **Cedar Creek Wealth Management** (Issaquah, WA)
    *   Analyzed investments to evaluate fund performance and recommend investments to financial advisors.
    *   Handled and preserved confidential data.
    *   Assisted in the entry and editing of personalized investment plans.
    *   Transferred money via wire, ACH, and journal.
    *   Answered client calls to assist with requests and issues.

*   **SULI Intern (Data Science)** at **Oak Ridge National Laboratory** (Oak Ridge, TN)
    *   Analyzed datasets of over 60,000 records to summarize fuel economy and usage in the United States using R.
    *   Employed clustering algorithms to more accurately estimate missing data.
    *   Collaborated with a mentor to decide on the best statistical methods to analyze data.
    *   Developed a research report and poster to communicate findings, which was presented at a po

In [8]:
rag_query("Which STAT classes has Cameron taken, and what were his grades in those classes?", documents, embeddings)

Cameron has taken the following STAT classes and received the indicated grades:

*   **STAT 2000** Statistical Methods: CR (Transfer Credit)
*   **STAT 3000** Statistics for Scientists: A
*   **STAT 5200** Analysis Designed Experiments: A
*   **STAT 5100** Modern Regression Methods: A
*   **STAT 5050** Introduction to R: A
*   **STAT 5645** Math Methods for Data Science: A
*   **STAT 5810** ST:Data Wrangling: A
*   **STAT 5555** Advanced R Prog for Data Science: A
*   **STAT 5685** Deep Learning Theory & App: A
*   **STAT 5810** ST: Applied Research in ML/AI: A


In [9]:
rag_query("What is causing rivers in the United States to get warmer?", documents, embeddings)

Rivers in the United States are getting warmer due to two main categories of causes:

1.  **Primary Drivers:** Climate change is identified as the primary driver of river warming.
2.  **Secondary Drivers:** These are human activities, specifically:
    *   The building of dams (especially large dams) for water preservation and power generation.
    *   Agriculture and its associated irrigation.

The study mentioned in the document found that the storage of water by dams had the greatest effect among the secondary drivers in creating longer heatwaves in rivers.


In [10]:
rag_query("How many pickup trucks are there in Pennsylvania?", documents, embeddings)

There are 1,496,337 pickup trucks registered in Pennsylvania.
